# RTX 5070 Ti YOLOv26m-cls Attention Screening Runner

Purpose: run the same **T4x2-safe** short YOLOv26m-cls attention screening configuration on a local RTX 5070 Ti machine.

This notebook:
- pulls the target GitHub branch without hardcoding secrets,
- optionally commits and pushes the current source changes after you review `git diff`,
- downloads `uynnhy/processed-images` using `kagglehub.dataset_download`,
- prepares the existing fixed split/YOLO dataset through the project scripts,
- runs baseline + attention screening with the same epoch value passed to list/validate/train/collect,
- fails loudly if there are no valid ranked results or the baseline is missing.

No GitHub token is embedded in this notebook. Use the `GITHUB_TOKEN` environment variable or paste it into the hidden prompt when asked.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap, time, platform, shutil, getpass, zipfile
from datetime import datetime, timezone

REPO_URL = "https://github.com/hntnhan1111-ayai/CVio_Shrimp_Disease_Classification_Capstone_SU26.git"
BRANCH = "feature/improving-lightweight-shrimp-disease-classification-coinfection-losses-randaugment"

# Local folder requested by user. Edit only if your actual folder is different.
REPO_DIR = Path(os.environ.get(
    "CVIO_SHRIMP_REPO_DIR",
    "~/notebooks/Improving Lightweight Shrimp Disease Classification with Co-Infection-Aware Losses and RandAugment"
)).expanduser()

OUTPUT_DIR = Path(os.environ.get(
    "YOLO_ATTENTION_OUTPUT_DIR",
    "~/notebooks/shrimp_outputs_yolo_attention_screening_rtx5070ti"
)).expanduser()
LOG_DIR = Path(os.environ.get(
    "NOTEBOOK_COMMAND_LOG_DIR",
    str(OUTPUT_DIR / "notebook_command_logs")
)).expanduser()

# Keep the same screening configuration as the Kaggle T4x2 runner; do not increase batch/workers/hyperparameters for RTX 5070 Ti.
YOLO_ATTENTION_SCREEN_EPOCHS = int(os.environ.get("YOLO_ATTENTION_SCREEN_EPOCHS", "10"))
KAGGLE_DATASET_ID = "uynnhy/processed-images"

RUN_INSTALL_MISSING = True
RUN_CLONE_OR_PULL = True
RUN_GIT_COMMIT_AND_PUSH = False  # Set True only after reviewing git diff in the commit cell.
RUN_DOWNLOAD_DATASET = True
RUN_PREPARE_DATASET = True
RUN_STATIC_VALIDATION = True
RUN_LIST_RUNS = True
RUN_VALIDATE_RESUME = True
RUN_ATTENTION_UNIT_TEST = True
RUN_MODEL_INJECTION_TEST = True
RUN_SHORT_SCREENING = True
RUN_COLLECT_RESULTS = True
RUN_ZIP_REPORTS = True

LOG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_ATTENTION_SCREEN_EPOCHS"] = str(YOLO_ATTENTION_SCREEN_EPOCHS)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Repo dir:", REPO_DIR)
print("Output dir:", OUTPUT_DIR)
print("Logs:", LOG_DIR)
print("Epochs:", YOLO_ATTENTION_SCREEN_EPOCHS)


In [ ]:
def _mask_secret_text(text: str) -> str:
    token = os.environ.get("GITHUB_TOKEN", "")
    if token:
        text = text.replace(token, "***GITHUB_TOKEN***")
    return text


def run_cmd(cmd, *, cwd=None, log_name="command.log", env=None, check=True):
    cwd = Path(cwd or Path.cwd())
    log_path = LOG_DIR / log_name
    full_env = os.environ.copy()
    if env:
        full_env.update({k: str(v) for k, v in env.items()})
    printable = " ".join(map(str, cmd))
    print(f"\n[RUN] {_mask_secret_text(printable)}")
    print(f"[CWD] {cwd}")
    print(f"[LOG] {log_path}")
    with open(log_path, "w", encoding="utf-8") as f:
        p = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            env=full_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert p.stdout is not None
        for line in p.stdout:
            safe_line = _mask_secret_text(line)
            print(safe_line, end="")
            f.write(safe_line)
        rc = p.wait()
        f.write(f"\n[returncode] {rc}\n")
    if check and rc != 0:
        raise RuntimeError(f"Command failed rc={rc}: {_mask_secret_text(printable)}. See {log_path}")
    return rc, log_path


def python_cmd(args, log_name, check=True):
    return run_cmd([sys.executable, *map(str, args)], cwd=REPO_DIR, log_name=log_name, check=check)


In [ ]:
# Environment and GPU check. This notebook is allowed to run on RTX 5070 Ti, but keeps T4x2-safe experiment settings.
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    print("cuda devices:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(i, torch.cuda.get_device_name(i))
except Exception as exc:
    print("torch unavailable:", repr(exc))

if sys.version_info[:2] != (3, 12):
    print("WARNING: Project was designed around Python 3.12.x. Continue only if your environment is compatible.")


In [ ]:
# Install only missing packages. Do not reinstall torch automatically; use the official PyTorch/CUDA wheel for your RTX 5070 Ti environment if torch is missing.
if RUN_INSTALL_MISSING:
    import importlib.util
    package_map = {
        "ultralytics": "ultralytics",
        "kagglehub": "kagglehub",
        "timm": "timm",
        "sklearn": "scikit-learn",
        "PIL": "pillow",
        "yaml": "pyyaml",
        "tqdm": "tqdm",
        "openpyxl": "openpyxl",
        "matplotlib": "matplotlib",
    }
    missing = [pkg for mod, pkg in package_map.items() if importlib.util.find_spec(mod) is None]
    print("missing packages:", missing)
    if importlib.util.find_spec("torch") is None:
        raise RuntimeError("torch is missing. Install the correct CUDA-enabled PyTorch build for this RTX 5070 Ti environment before continuing.")
    if missing:
        run_cmd([sys.executable, "-m", "pip", "install", *missing], log_name="000_install_missing.log")
else:
    print("Skipping package installation.")


In [ ]:
def get_github_token():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token:
        return token
    token = getpass.getpass("Paste GitHub token for this session only. It will not be saved in the notebook: ").strip()
    if not token:
        raise RuntimeError("Missing GitHub token. Set GITHUB_TOKEN or paste it into the hidden prompt.")
    os.environ["GITHUB_TOKEN"] = token
    return token


def git_auth_env():
    token = get_github_token()
    auth_dir = OUTPUT_DIR / "_runtime_auth"
    auth_dir.mkdir(parents=True, exist_ok=True)
    askpass = auth_dir / "git_askpass.sh"
    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
  *) echo "$GITHUB_TOKEN" ;;
esac
""",
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    return {"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"}


def clone_or_pull_repo():
    env = git_auth_env()
    if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
        run_cmd(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, env=env, log_name="010_git_fetch.log")
        run_cmd(["git", "checkout", BRANCH], cwd=REPO_DIR, env=env, log_name="011_git_checkout.log")
        run_cmd(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, env=env, log_name="012_git_pull.log")
    else:
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], cwd=REPO_DIR.parent, env=env, log_name="010_git_clone.log")
    run_cmd(["git", "status", "--short"], cwd=REPO_DIR, env=env, log_name="013_git_status_after_pull.log")

if RUN_CLONE_OR_PULL:
    clone_or_pull_repo()
else:
    print("Skipping clone/pull.")


In [ ]:
# Optional: commit and push source changes from the requested local folder.
# Keep this False until you inspect git status/diff. This stages only source/notebook files, not outputs/checkpoints/logs/datasets.
run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, log_name="014_git_branch.log", check=False)
run_cmd(["git", "status", "--short"], cwd=REPO_DIR, log_name="015_git_status_before_commit.log", check=False)
run_cmd(["git", "diff", "--", "shrimp_scripts", "experiments", "*.ipynb", "README.md"], cwd=REPO_DIR, log_name="016_git_diff_before_commit.log", check=False)

if RUN_GIT_COMMIT_AND_PUSH:
    env = git_auth_env()
    run_cmd(["git", "checkout", BRANCH], cwd=REPO_DIR, env=env, log_name="017_git_checkout_commit_branch.log")
    add_targets = [
        "shrimp_scripts/attention.py",
        "shrimp_scripts/models_yolo.py",
        "experiments/yolo_attention_screening",
        "kaggle_yolo26m_attention_screening_t4x2.ipynb",
        "rtx5070ti_yolo26m_attention_screening_runner.ipynb",
        "README.md",
    ]
    existing_targets = [t for t in add_targets if (REPO_DIR / t).exists()]
    if not existing_targets:
        raise RuntimeError("No expected source files found to stage. Check REPO_DIR.")
    run_cmd(["git", "add", *existing_targets], cwd=REPO_DIR, env=env, log_name="018_git_add.log")
    rc, _ = run_cmd(["git", "diff", "--cached", "--quiet"], cwd=REPO_DIR, env=env, log_name="019_git_cached_diff_quiet.log", check=False)
    if rc == 0:
        print("No staged source changes to commit.")
    else:
        run_cmd(["git", "commit", "-m", "Fix YOLO attention screening runner"], cwd=REPO_DIR, env=env, log_name="020_git_commit.log")
        run_cmd(["git", "push", "origin", BRANCH], cwd=REPO_DIR, env=env, log_name="021_git_push.log")
    run_cmd(["git", "status", "--short"], cwd=REPO_DIR, env=env, log_name="022_git_status_after_push.log")
    run_cmd(["git", "log", "--oneline", "-3"], cwd=REPO_DIR, env=env, log_name="023_git_log_after_push.log")
else:
    print("RUN_GIT_COMMIT_AND_PUSH is False. Review git status/diff first, then set it True if you want this notebook to commit and push.")


In [ ]:
# Download dataset with KaggleHub exactly once, then force project scripts to use that local downloaded path.
DATASET_PATH = None
if RUN_DOWNLOAD_DATASET:
    import kagglehub
    DATASET_PATH = Path(kagglehub.dataset_download(KAGGLE_DATASET_ID)).expanduser().resolve()
    print("Path to dataset files:", DATASET_PATH)
else:
    print("Skipping dataset download. Set DATASET_PATH manually if needed.")
    DATASET_PATH = Path(os.environ.get("SHRIMP_DATASET_ROOT", "")).expanduser().resolve()

if not DATASET_PATH or not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {DATASET_PATH}")

# Create a runtime-only kagglehub shim so run_01_prepare_dataset.py uses this already-downloaded path.
OVERRIDE_DIR = OUTPUT_DIR / "local_kagglehub_override"
OVERRIDE_DIR.mkdir(parents=True, exist_ok=True)
shim_source = f"""
def dataset_download(dataset_id):
    print('Using local KaggleHub override for', dataset_id)
    return r'{str(DATASET_PATH)}'
""".strip() + "\n"
(OVERRIDE_DIR / "kagglehub.py").write_text(shim_source, encoding="utf-8")
os.environ["PYTHONPATH"] = str(OVERRIDE_DIR) + os.pathsep + str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["SHRIMP_DATASET_ROOT"] = str(DATASET_PATH)
print("Runtime kagglehub override:", OVERRIDE_DIR / "kagglehub.py")


In [ ]:
# Validate required project files exist.
required_files = [
    REPO_DIR / "shrimp_scripts" / "models_yolo.py",
    REPO_DIR / "shrimp_scripts" / "attention.py",
    REPO_DIR / "shrimp_scripts" / "run_01_prepare_dataset.py",
    REPO_DIR / "experiments" / "yolo_attention_screening" / "run_yolo_attention_screen.py",
    REPO_DIR / "experiments" / "yolo_attention_screening" / "collect_yolo_attention_results.py",
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
print("Required files found.")


In [ ]:
# Save environment/version artifact.
def save_environment_versions():
    env = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "repo_dir": str(REPO_DIR),
        "output_dir": str(OUTPUT_DIR),
        "dataset_path": str(DATASET_PATH),
        "yolo_attention_screen_epochs": YOLO_ATTENTION_SCREEN_EPOCHS,
    }
    for mod_name in ["torch", "torchvision", "ultralytics", "kagglehub", "timm", "numpy", "pandas", "sklearn"]:
        try:
            mod = __import__(mod_name)
            env[mod_name] = getattr(mod, "__version__", "unknown")
        except Exception as exc:
            env[mod_name] = f"unavailable:{exc!r}"
    try:
        import torch
        env["cuda_available"] = torch.cuda.is_available()
        env["cuda_device_count"] = torch.cuda.device_count()
        env["cuda_devices"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception as exc:
        env["cuda_error"] = repr(exc)
    try:
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True).strip()
        env["git_commit"] = commit
        env["git_branch"] = subprocess.check_output(["git", "branch", "--show-current"], cwd=str(REPO_DIR), text=True).strip()
    except Exception as exc:
        env["git_error"] = repr(exc)
    path = OUTPUT_DIR / "environment_versions.json"
    path.write_text(json.dumps(env, indent=2, default=str), encoding="utf-8")
    print("Saved", path)
    return env

env_versions = save_environment_versions()
print(json.dumps(env_versions, indent=2, default=str)[:4000])


In [ ]:
# Static validation: compile source files.
if RUN_STATIC_VALIDATION:
    python_cmd(["-m", "compileall", "shrimp_scripts", "experiments"], "030_compileall.log")
else:
    print("Skipping static validation.")


In [ ]:
# Prepare fixed split + YOLO classification dataset through existing project script.
if RUN_PREPARE_DATASET:
    python_cmd(["shrimp_scripts/run_01_prepare_dataset.py", "--output_dir", str(OUTPUT_DIR), "--resume", "--progress"], "040_prepare_dataset.log")
else:
    print("Skipping dataset preparation.")


In [ ]:
# List planned attention runs.
if RUN_LIST_RUNS:
    python_cmd([
        "experiments/yolo_attention_screening/run_yolo_attention_screen.py",
        "--output_dir", str(OUTPUT_DIR),
        "--epochs", str(YOLO_ATTENTION_SCREEN_EPOCHS),
        "--list_runs",
    ], "050_attention_list_runs.log")


In [ ]:
# Validate resume/skip decisions before training.
if RUN_VALIDATE_RESUME:
    python_cmd([
        "experiments/yolo_attention_screening/run_yolo_attention_screen.py",
        "--output_dir", str(OUTPUT_DIR),
        "--epochs", str(YOLO_ATTENTION_SCREEN_EPOCHS),
        "--validate_resume",
    ], "051_attention_validate_resume.log")


In [ ]:
# Unit-test attention modules before modifying YOLO.
if RUN_ATTENTION_UNIT_TEST:
    python_cmd([
        "experiments/yolo_attention_screening/run_yolo_attention_screen.py",
        "--output_dir", str(OUTPUT_DIR),
        "--epochs", str(YOLO_ATTENTION_SCREEN_EPOCHS),
        "--self_test_attention_modules",
    ], "052_attention_unit_test.log")
else:
    print("Skipping attention module unit test.")


In [ ]:
# Optional but recommended: instantiate YOLOv26m-cls and verify attention injection before training.
if RUN_MODEL_INJECTION_TEST:
    python_cmd([
        "experiments/yolo_attention_screening/run_yolo_attention_screen.py",
        "--output_dir", str(OUTPUT_DIR),
        "--epochs", str(YOLO_ATTENTION_SCREEN_EPOCHS),
        "--self_test_model_injection",
    ], "053_attention_model_injection_test.log", check=False)
else:
    print("Skipping model injection test.")


In [ ]:
# Run the short attention screening. This can take hours for 9 YOLOv26m-cls runs.
if RUN_SHORT_SCREENING:
    python_cmd([
        "experiments/yolo_attention_screening/run_yolo_attention_screen.py",
        "--output_dir", str(OUTPUT_DIR),
        "--epochs", str(YOLO_ATTENTION_SCREEN_EPOCHS),
        "--resume",
        "--progress",
    ], "060_run_attention_screening.log", check=True)
else:
    print("Skipping training.")


In [ ]:
# Collect, rank, and fail loudly if there are no valid results or the baseline is missing.
if RUN_COLLECT_RESULTS:
    python_cmd([
        "experiments/yolo_attention_screening/collect_yolo_attention_results.py",
        "--output_dir", str(OUTPUT_DIR),
        "--epochs", str(YOLO_ATTENTION_SCREEN_EPOCHS),
    ], "070_collect_attention_results.log", check=True)
    import pandas as pd
    ranking_path = OUTPUT_DIR / "attention_ranking.csv"
    failed_path = OUTPUT_DIR / "attention_failed_or_skipped.csv"
    results_path = OUTPUT_DIR / "attention_screening_results.csv"
    if not ranking_path.exists():
        raise FileNotFoundError(ranking_path)
    ranking = pd.read_csv(ranking_path)
    failed = pd.read_csv(failed_path) if failed_path.exists() else pd.DataFrame()
    results = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()
    display(ranking)
    if not failed.empty:
        print("Failed/skipped/missing rows:")
        display(failed)
    if ranking.empty:
        raise RuntimeError("No valid completed attention runs. Do not interpret results.")
    if "none_baseline" not in set(ranking.get("attention_key", [])):
        raise RuntimeError("Baseline none_baseline is missing from valid ranking. Do not compare attention modules yet.")
else:
    print("Skipping collection.")


In [ ]:
# Zip report artifacts only, not checkpoints.
if RUN_ZIP_REPORTS:
    zip_path = OUTPUT_DIR.parent / "yolo_attention_screening_rtx5070ti_reports.zip"
    wanted_suffixes = {".csv", ".json", ".md", ".txt", ".log"}
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for base in [OUTPUT_DIR, LOG_DIR]:
            if not base.exists():
                continue
            for p in base.rglob("*"):
                if not p.is_file():
                    continue
                parts = set(p.parts)
                if "weights" in parts or "ultralytics_train" in parts:
                    continue
                if p.suffix.lower() in wanted_suffixes:
                    zf.write(p, p.relative_to(OUTPUT_DIR.parent))
    print("Created", zip_path, "size_MB", zip_path.stat().st_size / (1024 * 1024))
else:
    print("Skipping zip.")
